# SCENIC+ preprocessing — Patient 1 (03H096 / PB2)

**Role in the paper:** Prepares the RNA and ATAC inputs, the consensus peak
set and the pycisTopic topic models that feed eGRN inference.

**What this notebook does**
1. Splits the cleaned MultiVI object into an scRNA input for SCENIC+
2. Calls pseudobulk peaks per cell type with MACS2 and builds a consensus set
3. Runs pycisTopic barcode QC and builds the cisTopic object
4. Trains LDA topic models with Mallet and selects the 40-topic model
5. Binarises topics, calls differentially accessible regions per cell type and
   exports every region set as BED
6. Computes gene activity from accessibility

**Objects**
- **Reads:** `DATA_DIR / "01_Trimodal_integration_MultiVI/cleaned_MultiVI/MultiVI_Patient1_Relapse_03h096_Trimodal_Cleaned.h5ad"`,
  the ATAC fragments (`05_Gene_regulatory_networks/scATAC/atac_fragments.tsv.gz`)
  and the hg38 blacklist under `resources/`
- **Creates:** everything under
  `DATA_DIR / "05_Gene_regulatory_networks/PB2/outs/"` — pseudobulk BED/bigWig
  files, `consensus_regions.bed`, `cistopic_obj.pkl`, `models.pkl` and the
  `region_sets/` folder consumed by the SCENIC+ Snakemake pipeline

**Run this on a compute cluster, not a laptop.** Peak calling, barcode QC and
Mallet LDA (here with a large Mallet heap) are too heavy for a workstation.
Several cells were submitted as cluster jobs; the shell cells below are the
exact commands. After this notebook finishes, eGRN inference is the SCENIC+
Snakemake pipeline on the same cluster: copy
`05_Gene_regulatory_networks/config.yaml` over the Snakemake `config.yaml`
and run `snakemake --cores 20` (see the Snakemake notebook in this folder).

## Paths and settings

In [ ]:
from pathlib import Path

# Root of the companion data package. Point this at your local copy.
DATA_DIR = Path("PATH_TO_DATA")  # <-- set this to your local data root
OUT_DIR = DATA_DIR / "outputs/SCENICPLUS_PB2"     # figures and tables written by this notebook
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import os

os.environ['NUMBA_CACHE_DIR'] = '/tmp/'
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib-config"

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

In [ ]:
import scanpy as sc
import anndata as ad
import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import muon
import mudata as md
import scenicplus
import pybiomart

In [ ]:
sc.settings.set_figure_params(dpi=200, frameon=False)
sc.set_figure_params(dpi=200)
sc.set_figure_params(figsize=(20, 20))
torch.set_printoptions(precision=3, sci_mode=False, edgeitems=7)

## Load the cleaned MultiVI object and split the modalities

In [ ]:
adata = ad.read_h5ad(
    DATA_DIR / "01_Trimodal_integration_MultiVI/cleaned_MultiVI/MultiVI_Patient1_Relapse_03h096_Trimodal_Cleaned.h5ad"
)

In [ ]:
# Update the index of the AnnData object to remove labels after the underscore
adata.obs.index = adata.obs.index.str.split('_').str[0]

# Verify the updated index
print(adata.obs.index[:10])  # Display the first 10 cell names to confirm

In [ ]:
muon.pl.embedding(
    adata,
    basis="draw_graph_fa",
    color=["Cluster_Final"],
    frameon=False,
    size=200,
    ncols=1,
)

In [ ]:
# Extract gene expression data
scRNA_data = adata[:, adata.var['modality'] == 'Gene Expression']

# Extract peak data
scATAC_data = adata[:, adata.var['modality'] == 'Peaks']

In [ ]:
adata=scRNA_data.copy()
adata

## Normalise the RNA and select highly variable genes

In [ ]:
adata.raw = adata
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
adata = adata[:, adata.var.highly_variable]
sc.pp.scale(adata, max_value=10)

In [ ]:
# Check if 'highly_variable' exists
if 'highly_variable' in adata.var.columns:
    # Convert 'highly_variable' to NumPy boolean array if necessary
    adata.var['highly_variable'] = adata.var['highly_variable'].astype(bool).to_numpy()
else:
    print("The 'highly_variable' key does not exist in adata.var.")

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, log=True)

In [ ]:
sc.pl.draw_graph(adata, 
                 size=200,
                 color = 'Cluster_Final')

## Use the final clusters as the SCENIC+ `celltype` variable

In [ ]:
adata.obs['celltype'] = adata.obs['Cluster_Final']

In [ ]:
sc.pl.draw_graph(adata, size=200, color = 'celltype')

## Export the scRNA input for SCENIC+

In [ ]:
# Remove the 'protein_expression' entry from adata.obsm (not needed for SCENIC+ RNA)
if 'protein_expression' in adata.obsm:
    del adata.obsm['protein_expression']

# Add a '-PB2' suffix to cell barcodes so names stay unique across samples
adata.obs.index = adata.obs.index + '-PB2'

# Ensure every obsm matrix uses string-safe column labels where needed
for key, value in adata.obsm.items():
    if isinstance(value, pd.DataFrame):
        # Convert obsm column labels to strings if they are not already
        value.columns = value.columns.astype(str)
        adata.obsm[key] = pd.DataFrame(value, index=adata.obs.index)

# Save the AnnData object with the updated cell names
adata.write(os.path.join(work_dir, 'scRNA/adata.h5ad'), compression='gzip')

# Sanity check: print the first few cell names to confirm the suffix was applied
print(adata.obs.index[:5])

## pycisTopic: working directory and fragments

In [ ]:
import os
work_dir = DATA_DIR / "05_Gene_regulatory_networks/PB2"
import pycisTopic
#set some figure parameters for nice display inside jupyternotebooks.
%matplotlib inline

#make a directory for to store the processed scRNA-seq data.
if not os.path.exists(os.path.join(work_dir, 'scATAC')):
    os.makedirs(os.path.join(work_dir, 'scATAC'))
tmp_dir = './'

In [ ]:
fragments_dict = {'PB2': os.path.join(DATA_DIR / "05_Gene_regulatory_networks/scATAC/atac_fragments.tsv.gz")}

In [ ]:
import scanpy as sc
adata = sc.read_h5ad(os.path.join(work_dir, 'scRNA/adata.h5ad'))
adata.obs

In [ ]:
adata.obs.index = adata.obs.index.str.replace('-PB2', '')
adata.obs

In [ ]:
import scanpy as sc
cell_data = adata.obs
cell_data['sample_id'] = 'PB2'
cell_data['celltype'] = cell_data['celltype'].astype(str) # set data type of the celltype column to str, otherwise the export_pseudobulk function will complain.
del(adata)
cell_data

In [ ]:
import pycisTopic
pycisTopic.__version__

In [ ]:
import os
out_dir = DATA_DIR / "05_Gene_regulatory_networks/PB2/outs"
os.makedirs(out_dir, exist_ok = True)

## Pseudobulk peak calling per cell type

In [ ]:
chromsizes = pd.read_table(
    "http://hgdownload.cse.ucsc.edu/goldenPath/hg38/bigZips/hg38.chrom.sizes",
    header = None,
    names = ["Chromosome", "End"]
)
chromsizes.insert(1, "Start", 0)
chromsizes.head()

In [ ]:
from pycisTopic.pseudobulk_peak_calling import export_pseudobulk
os.makedirs(os.path.join(out_dir, "consensus_peak_calling"), exist_ok = True)
os.makedirs(os.path.join(out_dir, "consensus_peak_calling/pseudobulk_bed_files"), exist_ok = True)
os.makedirs(os.path.join(out_dir, "consensus_peak_calling/pseudobulk_bw_files"), exist_ok = True)


bw_paths, bed_paths = export_pseudobulk(
    input_data = cell_data,
    variable = "celltype",
    sample_id_col = "sample_id",
    chromsizes = chromsizes,
    bed_path = os.path.join(out_dir, "consensus_peak_calling/pseudobulk_bed_files"),
    bigwig_path = os.path.join(out_dir, "consensus_peak_calling/pseudobulk_bw_files"),
    path_to_fragments = fragments_dict,
    n_cpu = 10,
    normalize_bigwig = True,
    temp_dir = "./",
    split_pattern = "-"
)

In [ ]:
with open(os.path.join(out_dir, "consensus_peak_calling/bw_paths.tsv"), "wt") as f:
    for v in bw_paths:
        _ = f.write(f"{v}\t{bw_paths[v]}\n")

In [ ]:
with open(os.path.join(out_dir, "consensus_peak_calling/bed_paths.tsv"), "wt") as f:
    for v in bed_paths:
        _ = f.write(f"{v}\t{bed_paths[v]}\n")

In [ ]:
bw_paths = {}
with open(os.path.join(out_dir, "consensus_peak_calling/bw_paths.tsv")) as f:
    for line in f:
        v, p = line.strip().split("\t")
        bw_paths.update({v: p})

In [ ]:
bed_paths = {}
with open(os.path.join(out_dir, "consensus_peak_calling/bed_paths.tsv")) as f:
    for line in f:
        v, p = line.strip().split("\t")
        bed_paths.update({v: p})

## MACS2 peak calling and consensus peaks

In [ ]:
from pycisTopic.pseudobulk_peak_calling import peak_calling
macs_path = "macs2"

os.makedirs(os.path.join(out_dir, "consensus_peak_calling/MACS"), exist_ok = True)

narrow_peak_dict = peak_calling(
    macs_path = macs_path,
    bed_paths = bed_paths,
    outdir = os.path.join(os.path.join(out_dir, "consensus_peak_calling/MACS")),
    genome_size = 'hs',
    n_cpu = 10,
    input_format = 'BEDPE',
    shift = 73,
    ext_size = 146,
    keep_dup = 'all',
    q_value = 0.05,
)

In [ ]:
from pycisTopic.iterative_peak_calling import get_consensus_peaks
# Other param
peak_half_width=250
path_to_blacklist=DATA_DIR / "resources/hg38-blacklist.v2.bed"
# Get consensus peaks
consensus_peaks = get_consensus_peaks(
    narrow_peaks_dict = narrow_peak_dict,
    peak_half_width = peak_half_width,
    chromsizes = chromsizes,
    path_to_blacklist = path_to_blacklist)

In [ ]:
consensus_peaks.to_bed(
    path = os.path.join(out_dir, "consensus_peak_calling/consensus_regions.bed"),
    keep =True,
    compression = 'infer',
    chain = False)

## Transcription start sites

In [ ]:
!pycistopic tss gene_annotation_list | grep Human

In [ ]:
!mkdir -p {out_dir}/qc
!pycistopic tss get_tss \
    --output {out_dir}/qc/tss.bed \
    --name "hsapiens_gene_ensembl" \
    --to-chrom-source ucsc \
    --ucsc hg38

## Barcode-level ATAC quality control

This step is long-running and was submitted as a batch job; the command below is the exact one used.

In [ ]:
fragments_file = fragments_dict["PB2"]

!pycistopic qc \
    --fragments {fragments_file} \
    --regions {out_dir}/consensus_peak_calling/consensus_regions.bed \
    --tss {out_dir}/qc/tss.bed \
    --output {out_dir}/qc/PB2

## QC plots and barcode filtering

In [ ]:
from pycisTopic.plotting.qc_plot import plot_sample_stats, plot_barcode_stats
import matplotlib.pyplot as plt

In [ ]:
for sample_id in fragments_dict:
    fig = plot_sample_stats(
        sample_id=sample_id,
        pycistopic_qc_output_dir=f"{DATA_DIR / '05_Gene_regulatory_networks/PB2/outs/qc'}/"
    )

In [ ]:
from pycisTopic.qc import get_barcodes_passing_qc_for_sample
sample_id_to_barcodes_passing_filters = {}
sample_id_to_thresholds = {}
for sample_id in fragments_dict:
    (
        sample_id_to_barcodes_passing_filters[sample_id],
        sample_id_to_thresholds[sample_id]
    ) = get_barcodes_passing_qc_for_sample(
            sample_id = sample_id,
            pycistopic_qc_output_dir = DATA_DIR / "05_Gene_regulatory_networks/PB2/outs/qc",
            unique_fragments_threshold = None, # use automatic thresholding
            tss_enrichment_threshold = None, # use automatic thresholding
            frip_threshold = 0,
            use_automatic_thresholds = True,
    )

In [ ]:
for sample_id in fragments_dict:
    fig = plot_barcode_stats(
        sample_id = sample_id,
        pycistopic_qc_output_dir = DATA_DIR / "05_Gene_regulatory_networks/PB2/outs/qc",
        bc_passing_filters = sample_id_to_barcodes_passing_filters[sample_id],
        detailed_title = False,
        **sample_id_to_thresholds[sample_id]
    )

## Build the cisTopic object

In [ ]:
path_to_regions = os.path.join(out_dir, "consensus_peak_calling/consensus_regions.bed")
path_to_blacklist = DATA_DIR / "resources/hg38-blacklist.v2.bed"
pycistopic_qc_output_dir = DATA_DIR / "05_Gene_regulatory_networks/PB2/outs/qc"

from pycisTopic.cistopic_class import create_cistopic_object_from_fragments
import polars as pl

cistopic_obj_list = []
for sample_id in fragments_dict:
    sample_metrics = pl.read_parquet(
        os.path.join(pycistopic_qc_output_dir, f'{sample_id}.fragments_stats_per_cb.parquet')
    ).to_pandas().set_index("CB").loc[sample_id_to_barcodes_passing_filters[sample_id]]
    cistopic_obj = create_cistopic_object_from_fragments(
        path_to_fragments = fragments_dict[sample_id],
        path_to_regions = path_to_regions,
        path_to_blacklist = path_to_blacklist,
        metrics = sample_metrics,
        valid_bc = sample_id_to_barcodes_passing_filters[sample_id],
        n_cpu = 1,
        project = sample_id,
        split_pattern = '-'
    )
    cistopic_obj_list.append(cistopic_obj)

In [ ]:
cistopic_obj = cistopic_obj_list[0]
print(cistopic_obj)

In [ ]:
import pickle
pickle.dump(
    cistopic_obj,
    open(os.path.join(out_dir, "cistopic_obj.pkl"), "wb")
)

In [ ]:
import pickle
import os

# Loading the serialized object from the file using the existing 'out_dir'
with open(os.path.join(out_dir, "cistopic_obj.pkl"), 'rb') as file:
    cistopic_obj = pickle.load(file)

## Topic modelling with Mallet

In [ ]:
# Mallet provides the collapsed Gibbs sampler used for the LDA models.
MALLET_DIR = DATA_DIR / "resources/Mallet-202108"

!wget https://github.com/mimno/Mallet/releases/download/v202108/Mallet-202108-bin.tar.gz
!tar -xf Mallet-202108-bin.tar.gz

In [ ]:
!mkdir -p {MALLET_DIR}/tutorial

In [ ]:
!mkdir -p {MALLET_DIR}/bin/mallet

In [ ]:
os.environ['MALLET_MEMORY'] = '200G'
from pycisTopic.lda_models import run_cgs_models_mallet
# Configure path Mallet
mallet_path = str(MALLET_DIR / "bin/mallet")
# Run models
models=run_cgs_models_mallet(
    cistopic_obj,
    n_topics=[2, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50],
    n_cpu=12,
    n_iter=500,
    random_state=555,
    alpha=50,
    alpha_by_topic=True,
    eta=0.1,
    eta_by_topic=False,
    tmp_path=DATA_DIR / "resources/Mallet-202108/tutorial",
    save_path=DATA_DIR / "resources/Mallet-202108/tutorial",
    mallet_path=mallet_path,
)

In [ ]:
pickle.dump(
    models,
    open(os.path.join(out_dir, "models.pkl"), "wb")
)

## Optional: skip training and load the saved topic models

If `models.pkl` already exists, load it instead of re-running the Mallet models above.

In [ ]:
import pickle
import os

# Loading the serialized object from the file using the existing 'out_dir'
with open(os.path.join(out_dir, "models.pkl"), 'rb') as file:
    models = pickle.load(file)

## Select the topic model and add it to the cisTopic object

In [ ]:
from pycisTopic.lda_models import evaluate_models
model = evaluate_models(
    models,
    select_model = 40,   # number of topics retained for this patient
    return_model = True
)

In [ ]:
cistopic_obj.add_LDA_model(model)

## Clustering and visualisation of the topic space

In [ ]:
from pycisTopic.clust_vis import (
    find_clusters,
    run_umap,
    run_tsne,
    plot_metadata,
    plot_topic,
    cell_topic_heatmap
)

In [ ]:
find_clusters(
    cistopic_obj,
    target  = 'cell',
    k = 10,
    res = [0.6, 1.2, 3],
    prefix = 'pycisTopic_',
    scale = True,
    split_pattern = '-'
)

In [ ]:
run_tsne(
    cistopic_obj,
    target  = 'cell', scale=True)

## Transfer the ForceAtlas2 embedding onto the cisTopic object

In [ ]:
import os
import pandas as pd
import scanpy as sc

# Step 1: Load the .h5ad object
work_dir = f"{DATA_DIR / '05_Gene_regulatory_networks/PB2'}/"
h5ad_file_path = os.path.join(work_dir, 'scRNA/adata.h5ad')

# Load the h5ad object
h5ad_obj = sc.read_h5ad(h5ad_file_path)

# Step 2: Extract the FA embedding and celltype information from the .h5ad object
fa_embedding = h5ad_obj.obsm['X_draw_graph_fa']  # FA embedding
h5ad_cell_names = h5ad_obj.obs.index  # Cell names
h5ad_celltype = h5ad_obj.obs['celltype']  # Celltype column

# Step 3: Create a DataFrame for FA embedding and celltype
h5ad_metadata = pd.DataFrame({
    'FA_1': fa_embedding[:, 0],
    'FA_2': fa_embedding[:, 1],
    'celltype': h5ad_celltype
}, index=h5ad_cell_names)

# Step 4: Get the cell names from the CistopicObject (cistopic_obj)
pkl_cell_names = pd.Index(cistopic_obj.cell_names)  # Cell names from the CistopicObject

# Step 5: Find the common cells between the h5ad cell names and the CistopicObject cell names
common_cells = h5ad_metadata.index.intersection(pkl_cell_names)

# Step 6: Align the FA projection and celltype metadata with the matched cell names
# Filter the metadata to include only the common cells
h5ad_metadata_matched = h5ad_metadata.loc[common_cells]

# Step 7: Reorder the cell names in the CistopicObject to match the metadata
cell_names_ordered = pkl_cell_names[pkl_cell_names.isin(common_cells)]

# Step 8: Add the aligned FA projection to the CistopicObject
cistopic_obj.projections['cell']['X_draw_graph_fa'] = h5ad_metadata_matched.loc[cell_names_ordered, ['FA_1', 'FA_2']]

# Step 9: Add the celltype metadata to the CistopicObject
if not hasattr(cistopic_obj, 'cell_data'):
    cistopic_obj.cell_data = pd.DataFrame(index=cistopic_obj.cell_names)

cistopic_obj.cell_data['celltype'] = h5ad_metadata_matched.loc[cell_names_ordered, 'celltype']

# Verify the updated metadata
print(cistopic_obj.cell_data.head())

# Step 10: Plot the metadata with the aligned FA projection
plot_metadata(
    cistopic_obj,
    reduction_name='X_draw_graph_fa',
    variables=['celltype', 'pycisTopic_leiden_10_0.6', 'pycisTopic_leiden_10_1.2', 'pycisTopic_leiden_10_3'],
    target='cell',
    num_columns=4,
    text_size=10,
    dot_size=5
)

In [ ]:
pickle.dump(
    cistopic_obj,
    open(os.path.join(out_dir, "cistopic_obj.pkl"), "wb")
)

In [ ]:
# Now, run the plot_metadata again
plot_metadata(
    cistopic_obj,
    reduction_name='X_draw_graph_fa',
    variables=['celltype'],
    target='cell',
    num_columns=4,
    text_size=10,
    dot_size=5
)

## Label the pycisTopic clusters with the dominant cell type

In [ ]:
annot_dict = {}
for resolution in [0.6, 1.2, 3]:
    annot_dict[f"pycisTopic_leiden_10_{resolution}"] = {}
    for cluster in set(cistopic_obj.cell_data[f"pycisTopic_leiden_10_{resolution}"]):
        counts = cistopic_obj.cell_data.loc[
            cistopic_obj.cell_data[f"pycisTopic_leiden_10_{resolution}"] == cluster,
            "celltype"].value_counts()
        
        # Check if 'counts' is empty before attempting to find the argmax
        if not counts.empty:
            # Get the most frequent cell type and annotate it with its cluster
            most_frequent_celltype = counts.idxmax()  # Using idxmax as a safer alternative to counts.index[counts.argmax()]
            annot_dict[f"pycisTopic_leiden_10_{resolution}"][cluster] = f"{most_frequent_celltype}({cluster})"
        else:
            # Handle the case where no cell types are available for this cluster
            annot_dict[f"pycisTopic_leiden_10_{resolution}"][cluster] = f"Unknown({cluster})"

In [ ]:
for resolution in [0.6, 1.2, 3]:
    cistopic_obj.cell_data[f'pycisTopic_leiden_10_{resolution}'] = [
        annot_dict[f'pycisTopic_leiden_10_{resolution}'][x] for x in cistopic_obj.cell_data[f'pycisTopic_leiden_10_{resolution}'].tolist()
    ]

In [ ]:
plot_metadata(
    cistopic_obj,
    reduction_name='X_draw_graph_fa',
    variables=['celltype', 'pycisTopic_leiden_10_0.6', 'pycisTopic_leiden_10_1.2', 'pycisTopic_leiden_10_3'],
    target='cell', num_columns=4,
    text_size=10,
    dot_size=5)

In [ ]:
plot_metadata(
    cistopic_obj,
    reduction_name='X_draw_graph_fa',
    variables=['log10_unique_fragments_count', 'tss_enrichment', 
              #'Doublet_scores_fragments', 
               'fraction_of_fragments_in_peaks'],
    target='cell', num_columns=4,
    text_size=10,
    dot_size=5)

In [ ]:
plot_topic(
    cistopic_obj,
    reduction_name = 'X_draw_graph_fa',
    target = 'cell',
    num_columns=5
)

## Binarise topics

In [ ]:
from pycisTopic.topic_binarization import binarize_topics

In [ ]:
region_bin_topics_top_3k = binarize_topics(
    cistopic_obj, method='ntop', ntop = 3_000,
    plot=True, num_columns=5
)

In [ ]:
region_bin_topics_otsu = binarize_topics(
    cistopic_obj, method='otsu',
    plot=True, num_columns=5
)

In [ ]:
binarized_cell_topic = binarize_topics(
    cistopic_obj,
    target='cell',
    method='li',
    plot=True,
    num_columns=5, nbins=100)

## Topic quality control and annotation

In [ ]:
from pycisTopic.topic_qc import compute_topic_metrics, plot_topic_qc, topic_annotation
import matplotlib.pyplot as plt
from pycisTopic.utils import fig2img

In [ ]:
topic_qc_metrics = compute_topic_metrics(cistopic_obj)

In [ ]:
fig_dict={}
fig_dict['CoherenceVSAssignments']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Log10_Assignments', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['AssignmentsVSCells_in_bin']=plot_topic_qc(topic_qc_metrics, var_x='Log10_Assignments', var_y='Cells_in_binarized_topic', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSCells_in_bin']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Cells_in_binarized_topic', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSRegions_in_bin']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Regions_in_binarized_topic', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSMarginal_dist']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Marginal_topic_dist', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSGini_index']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Gini_index', var_color='Gini_index', plot=False, return_fig=True)

In [ ]:
topic_annot = topic_annotation(
    cistopic_obj,
    annot_var='celltype',
    binarized_cell_topic=binarized_cell_topic,
    general_topic_thr = 0.2
)

## Impute accessibility and call differentially accessible regions

In [ ]:
from pycisTopic.diff_features import (
    impute_accessibility,
    normalize_scores,
    find_highly_variable_features,
    find_diff_features
)
import numpy as np

In [ ]:
imputed_acc_obj = impute_accessibility(
    cistopic_obj,
    selected_cells=None,
    selected_regions=None,
    scale_factor=10**6
)

In [ ]:
normalized_imputed_acc_obj = normalize_scores(imputed_acc_obj, scale_factor=10**4)

In [ ]:
variable_regions = find_highly_variable_features(
    normalized_imputed_acc_obj,
    min_disp = 0.05,
    min_mean = 0.0125,
    max_mean = 3,
    max_disp = np.inf,
    n_bins=20,
    n_top_features=None,
    plot=True
)

In [ ]:
markers_dict= find_diff_features(
    cistopic_obj,
    imputed_acc_obj,
    variable='celltype',
    var_features=variable_regions,
    contrasts=None,
    adjpval_thr=0.05,
    log2fc_thr=np.log2(1.2),
    n_cpu=5,
    split_pattern = '-'
)

In [ ]:
from pycisTopic.clust_vis import plot_imputed_features

In [ ]:
print("Number of DARs found:")
print("---------------------")
for x in markers_dict:
    print(f"  {x}: {len(markers_dict[x])}")

## Export every region set as BED for the SCENIC+ pipeline

In [ ]:
os.makedirs(os.path.join(out_dir, "region_sets"), exist_ok = True)
os.makedirs(os.path.join(out_dir, "region_sets", "Topics_otsu"), exist_ok = True)
os.makedirs(os.path.join(out_dir, "region_sets", "Topics_top_3k"), exist_ok = True)
os.makedirs(os.path.join(out_dir, "region_sets", "DARs_cell_type"), exist_ok = True)

In [ ]:
from pycisTopic.utils import region_names_to_coordinates

In [ ]:
for topic in region_bin_topics_otsu:
    region_names_to_coordinates(
        region_bin_topics_otsu[topic].index
    ).sort_values(
        ["Chromosome", "Start", "End"]
    ).to_csv(
        os.path.join(out_dir, "region_sets", "Topics_otsu", f"{topic}.bed"),
        sep = "\t",
        header = False, index = False
    )

In [ ]:
for topic in region_bin_topics_top_3k:
    region_names_to_coordinates(
        region_bin_topics_top_3k[topic].index
    ).sort_values(
        ["Chromosome", "Start", "End"]
    ).to_csv(
        os.path.join(out_dir, "region_sets", "Topics_top_3k", f"{topic}.bed"),
        sep = "\t",
        header = False, index = False
    )

In [ ]:
for cell_type in markers_dict:
    region_names_to_coordinates(
        markers_dict[cell_type].index
    ).sort_values(
        ["Chromosome", "Start", "End"]
    ).to_csv(
        os.path.join(out_dir, "region_sets", "DARs_cell_type", f"{cell_type}.bed"),
        sep = "\t",
        header = False, index = False
    )

## Gene activity from accessibility

In [ ]:
import pyranges as pr
from pycisTopic.gene_activity import get_gene_activity

In [ ]:
chromsizes = pd.read_table(os.path.join(out_dir, "qc", "hg38.chrom_sizes_and_alias.tsv"))
chromsizes

In [ ]:
chromsizes.rename({"# ucsc": "Chromosome", "length": "End"}, axis = 1, inplace = True)
chromsizes["Start"] = 0
chromsizes = pr.PyRanges(chromsizes[["Chromosome", "Start", "End"]])

In [ ]:
pr_annotation = pd.read_table(
        os.path.join(out_dir, "qc", "tss.bed")
    ).rename(
        {"Name": "Gene", "# Chromosome": "Chromosome"}, axis = 1)
pr_annotation["Transcription_Start_Site"] = pr_annotation["Start"]
pr_annotation = pr.PyRanges(pr_annotation)
pr_annotation

In [ ]:
gene_act, weigths = get_gene_activity(
    imputed_acc_obj,
    pr_annotation,
    chromsizes,
    use_gene_boundaries=True, # Whether to use the whole search space or stop when encountering another gene
    upstream=[1000, 100000], # Search space upstream. The minimum means that even if there is a gene right next to it
                             # these bp will be taken (1kbp here)
    downstream=[1000,100000], # Search space downstream
    distance_weight=True, # Whether to add a distance weight (an exponential function, the weight will decrease with distance)
    decay_rate=1, # Exponent for the distance exponential funciton (the higher the faster will be the decrease)
    extend_gene_body_upstream=10000, # Number of bp upstream immune to the distance weight (their value will be maximum for
                          #this weight)
    extend_gene_body_downstream=500, # Number of bp downstream immune to the distance weight
    gene_size_weight=False, # Whether to add a weights based on the length of the gene
    gene_size_scale_factor='median', # Dividend to calculate the gene size weigth. Default is the median value of all genes
                          #in the genome
    remove_promoters=False, # Whether to remove promoters when computing gene activity scores
    average_scores=True, # Whether to divide by the total number of region assigned to a gene when calculating the gene
                          #activity score
    scale_factor=1, # Value to multiply for the final gene activity matrix
    extend_tss=[10,10], # Space to consider a promoter
    gini_weight = True, # Whether to add a gini index weigth. The more unique the region is, the higher this weight will be
    return_weights= True, # Whether to return the final weights
    project='Gene_activity') # Project name for the gene activity object

In [ ]:
DAG_markers_dict= find_diff_features(
    cistopic_obj,
    gene_act,
    variable='celltype',
    var_features=None,
    contrasts=None,
    adjpval_thr=0.05,
    log2fc_thr=np.log2(1.2),
    n_cpu=5,
    split_pattern = '-')

In [ ]:
from pycisTopic.loom import export_region_accessibility_to_loom, export_gene_activity_to_loom

In [ ]:
cluster_markers = {'celltype': markers_dict}